In [ ]:
import boto3
import copy

In [ ]:
print(boto3.__version__)

In [ ]:
# first we will find out which user or role is being used by sagemaker
# We will assign BedrockFUllAccess to it.
session = boto3.Session()
sts_client = session.client('sts')
identity = sts_client.get_caller_identity()
print(identity['Arn'])

In [ ]:
# import kb_id from my private file config.py
from config import *

In [ ]:
kb_id = kb_id
region_name = 'us-east-1'
model_id =  ["Claude 3 Sonnet", "anthropic.claude-3-sonnet-20240229-v2:0"]
model_arn = 'arn:aws:bedrock:us-east-1::anthropic.claude-3-sonnet-20240620-v1'

In [ ]:
bedrock = boto3.client(service_name='bedrock')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime', region_name = region_name)

In [ ]:
# we will create a small retrieveAndGenerateConfiguration that I know works
retrieveAndGenerateConfiguration={
    'type': 'KNOWLEDGE_BASE',
    'knowledgeBaseConfiguration': {
        'knowledgeBaseId': kb_id,
        'modelArn': model_arn,
        'generationConfiguration': {
            'promptTemplate': {
                'textPromptTemplate': '''<instruction>Here are the search results from the knowledge base: $search_results$
                Study these search results carefully and identify the different question formats present. Choose one of those formats and use it to formulate your next question. Respond with the question you formulated, without any additional preamble or explanation. Answer as accurately as possible!</instruction>'''
            }
        },
    }
}
query = 'list repeated packages, associate them with their source file'

In [ ]:
error1_config = copy.deepcopy(retrieveAndGenerateConfiguration)
e1_model_arns = ['arn:aws:bedrock:us-east-1::anthropic.claude-3-sonnet-20240620-v1','arn:aws:bedrock:us-east-1::anthropic.claude-3-5-sonnet-20241022-v2:0']


# Add the orchestrationConfiguration
error1_config['knowledgeBaseConfiguration']['orchestrationConfiguration'] = {
    'queryTransformationConfiguration': {
        'type': 'QUERY_DECOMPOSITION',
    }
}

for m in e1_model_arns:
    error1_config['knowledgeBaseConfiguration']['modelArn'] = m
    try:
        response = bedrock_agent_runtime_client.retrieve_and_generate(
            input={'text': query},
            retrieveAndGenerateConfiguration=error1_config
        )
    except Exception as e:
        print(f"Exception occurred for model {m}:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")

In [ ]:
# the second error
error2_config = copy.deepcopy(retrieveAndGenerateConfiguration)
for m in e1_model_arns:
    error2_config['knowledgeBaseConfiguration']['modelArn'] = m
    try:
        response = bedrock_agent_runtime_client.retrieve_and_generate(
            input={'text': query},
            retrieveAndGenerateConfiguration=error2_config
        )
    except Exception as e:
        print(f"Exception occurred for model {m}:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")

In [ ]:
error3_config = copy.deepcopy(retrieveAndGenerateConfiguration)

error3_config['knowledgeBaseConfiguration']['orchestrationConfiguration'] = {
    'queryTransformationConfiguration': {
        'type': 'QUERY_DECOMPOSITION',
        'promptTemplate': {
            'textPromptTemplate': '<instruction>You are a question formulation assistant. Review the conversation history: $conversation_history$ Your task is to: 1. Analyze the conversation context 2. Break down complex queries into simpler components 3. Ensure questions align with the knowledge base content $output_format_instructions$ Generate a focused search query that will help find relevant question formats in the knowledge base.</instruction>'
        }
    }
}

for m in e1_model_arns:
    error2_config['knowledgeBaseConfiguration']['modelArn'] = m
    try:
        response = bedrock_agent_runtime_client.retrieve_and_generate(
            input={'text': query},
            retrieveAndGenerateConfiguration=error3_config
        )
    except Exception as e:
        print(f"Exception occurred for model {m}:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")

In [ ]:
# now I will provide a successful example

In [ ]:
# first we correct the model names, Here is a correct list of them
for model in bedrock.list_foundation_models()['modelSummaries']:
    print(model['modelId'])

In [ ]:
# Note that models that end with k usually are for provisioned throughput. Since we are using on-demand we can use the ones that end in 0
models = ['anthropic.claude-3-sonnet-20240229-v1:0']

In [ ]:
# a corrected configuration per the boto3 documentation in https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html

# Important!: There is no promptTemplate under queryTransformationConfiguration, but under orchestrationConfiguration
retrieveAndGenerateCorrectedConfiguration = {
    'type': 'KNOWLEDGE_BASE',
    'knowledgeBaseConfiguration': {
        'knowledgeBaseId': kb_id,
        'generationConfiguration': {
            'promptTemplate': {
                'textPromptTemplate': '''<instruction>Here are the search results from the knowledge base: $search_results$
                Study these search results carefully and Answer as accurately as possible!</instruction>'''
            }
        },
        'orchestrationConfiguration': {
            'queryTransformationConfiguration': {
                'type': 'QUERY_DECOMPOSITION',
            },
            'promptTemplate': {
                'textPromptTemplate': '''<instruction>You are an assistant. Review the conversation history: $conversation_history$ Your task is to:
    1. Analyze the conversation context
    2. Break down complex queries into simpler components
    3. Ensure questions align with the knowledge base content
    $output_format_instructions$
    Generate a focused search query that will help find relevant information from the knowledge base.</instruction>'''
            }
        }
    }
}




In [ ]:
for m in models:
    retrieveAndGenerateCorrectedConfiguration['knowledgeBaseConfiguration']['modelArn'] = m
    try:
        response = bedrock_agent_runtime_client.retrieve_and_generate(
            input={'text': query},
            retrieveAndGenerateConfiguration=retrieveAndGenerateCorrectedConfiguration
        )
        generated_text = response['output']['text']
        print(generated_text)
    except Exception as e:
        print(f"Exception occurred for model {m}:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")